In [ ]:
import pandas as pd
df=pd.read_csv("train.csv")
test_df=pd.read_csv("test.csv")
df

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
na_columns = ["feature_wind", "feature_humidity"]
df[na_columns] = df.groupby(["Month", "Hour", "category_weather"])[
    na_columns
].transform(lambda x: x.fillna(x.mean()))
df[na_columns] = df.groupby(["Month", "Hour"])[na_columns].transform(lambda x: x.fillna(x.mean()))
df[na_columns] = df[na_columns].fillna(df[na_columns].median())
df["Month"] = df["Month"].astype(str)
df["season_code"] = df["season_code"].astype(str)
df["category_weather"] = df["category_weather"].astype(str)
df["hour_day_cat"] = (
    "H" + df["Hour"].astype(str) + "_D" + df["Day_of_Week"].astype(str))
base_features = ["Year","feature_feel_temp","feature_humidity","feature_wind","flag_holiday"]
categorical_cols = ["hour_day_cat", "Month", "season_code", "category_weather"]
X_train_raw = df[base_features + categorical_cols].copy()
X_train_final = pd.get_dummies(X_train_raw, columns=categorical_cols, drop_first=True)
y_train_log = np.log1p(df["demand"])


test_df[na_columns] = test_df.groupby(["Month", "Hour", "category_weather"])[na_columns].transform(lambda x: x.fillna(x.mean()))
test_df[na_columns] = test_df.groupby(["Month", "Hour"])[na_columns].transform(lambda x: x.fillna(x.mean()))
test_df[na_columns] = test_df[na_columns].fillna(df[na_columns].median())
test_df["Month"] = test_df["Month"].astype(str)
test_df["season_code"] = test_df["season_code"].astype(str)
test_df["category_weather"] = test_df["category_weather"].astype(str)
test_df["hour_day_cat"] = ("H" + test_df["Hour"].astype(str) + "_D" + test_df["Day_of_Week"].astype(str))
X_test_raw = test_df[base_features + categorical_cols].copy()
X_test_encoded = pd.get_dummies(X_test_raw, columns=categorical_cols, drop_first=True)
X_test_final = X_test_encoded.reindex(columns=X_train_final.columns, fill_value=0)

final_submission_model = Ridge(alpha=1.0)
final_submission_model.fit(X_train_final, y_train_log)
test_predictions_log = final_submission_model.predict(X_test_final)
test_predictions = np.expm1(test_predictions_log)
test_predictions = np.clip(test_predictions, 0, None)

submission = pd.DataFrame({"id": test_df["id"],"demand": test_predictions, })
submission.to_csv("final_submission.csv", index=False)
print("Pipeline complete! 'final_submission.csv' has been generated with all unpacked categorical indicators.")